In [3]:
import sys
import numpy as np
import importlib

sys.path.append('../src')
import policies 
import bbDebiasing2
import wbDebiasing


## Black Box Algorithm Tests

Sanity check: if all my initial policies have single LS, should end up w predictor of mean.

In [58]:
importlib.reload(bbDebiasing2)
curr_preds = np.zeros([3,2])
my_policy = policies.Simplex(2)
other_policies = [np.array([[1,0],[1,0],[1,0]])] #debiasing wrt these LS will just give you average predictor
train_ys = np.array([[1,0],[2,-1],[3,5]])
tolerance=0.1

bbModeltest = bbDebiasing2.bbModel(my_policy, other_policies,train_ys, curr_preds, tolerance)
print(bbModeltest.debias()==np.tile(train_ys.mean(axis=0), (len(train_ys),1)))
bbModeltest.predict(curr_preds, other_policies)==np.tile(train_ys.mean(axis=0), (len(train_ys),1))

[[ True  True]
 [ True  True]
 [ True  True]]


array([[ True,  True],
       [ True,  True],
       [ True,  True]])

Sanity check: if initial policies have $n$ disjoint level sets, should converge towards exact labels. Note: they don't go exactly to this because the maximal level set in each round often is the one where the policy is 0 in multiple coordinates, so do end up iterating through LSs over many rounds until tolerance constraint is met. 

In [57]:
importlib.reload(bbDebiasing2)
curr_preds = np.zeros([3,2])
my_policy = policies.Simplex(2)
other_policies = [np.array([[1,0],[0,1],[0,0]])] #debiasing wrt these LS will just give you average predictor
train_y = np.array([[1,0],[2,-1],[3,5]])
tolerance=0.01

bbModeltest2 = bbDebiasing2.bbModel(my_policy, other_policies,train_ys,curr_preds, tolerance)
print(bbModeltest2.debias())
bbModeltest2.predict(curr_preds, other_policies)==bbModeltest2.debias()

[[ 1.0078125   0.01953125]
 [ 2.         -1.        ]
 [ 2.9921875   4.98046875]]


array([[ True,  True],
       [ True,  True],
       [ True,  True]])

## Whitebox Algorithm Tests

In [35]:
k,d,n = 2,3,5

all_policies = [policies.Simplex(d) for i in range(k)]
np.random.seed(42)
train_ys = np.random.binomial(1,0.5,(n,d)).astype(np.float64)
preds_by_models = np.random.binomial(1,0.5,(k,n,d)).astype(np.float64)
tolerance = 0.1

In [36]:
importlib.reload(wbDebiasing)
wbModel = wbDebiasing.wbModel(all_policies, train_ys, preds_by_models, tolerance)
out = wbModel.debias()

In [37]:
np.array(wbModel.mses_by_round)[:,1]

array([[0.6       , 0.8       , 0.6       ],
       [0.6       , 0.8       , 0.6       ],
       [0.6       , 0.8       , 0.6       ],
       [0.6       , 0.8       , 0.53333333],
       [0.6       , 0.8       , 0.53333333],
       [0.4       , 0.6       , 0.53333333],
       [0.4       , 0.4       , 0.53333333],
       [0.4       , 0.4       , 0.53333333],
       [0.4       , 0.2       , 0.17777778],
       [0.3       , 0.2       , 0.13333333],
       [0.1       , 0.        , 0.04444444],
       [0.075     , 0.        , 0.03333333],
       [0.01875   , 0.        , 0.00833333]])

In [738]:
# note: in order to be runable, have to store preds
# oos_out, oos_rounds = wbModel.predict(preds_by_models)
# for i in range(len(oos_rounds)):
#     if (oos_rounds[i]!=wbModel.preds_by_rounds[i]).any():
#         print(i)

In [38]:
ensemble_policy, ensemble_model, expected_self_eval = wbModel.ensemble(out)
expected_self_eval

np.float64(0.9944444444444445)

In [39]:
wbModel.calc_ensemble_return(ensemble_policy, train_ys)

(np.float64(1.0), array([1., 1., 1., 1., 1.]))

In [40]:
train_ys

array([[0., 1., 1.],
       [1., 0., 0.],
       [0., 1., 1.],
       [1., 0., 1.],
       [1., 0., 0.]])

In [41]:
ensemble_policy

array([[0., 1., 0.],
       [1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       [1., 0., 0.]])

# Folktables Pipeline Testing

In [12]:
sys.path.append('folktables-pipeline')
import acsData
import folktablesModelSetup as setup

importlib.reload(setup)

acs_year = '2016'
state_names = ['CA', 'OR', 'AK']
state_list = setup.runSetup(acs_year, state_names)

importlib.reload(setup)
ensemble = setup.Ensemble(state_list)
ensemble_meta_preds = ensemble.meta_predictor(ensemble.features_test)
ensemble_preds = ensemble.predict(ensemble.features_test)

In [25]:
meta_preds_verification = np.zeros((len(ensemble.features_test), ensemble.n_predictors))
for i,state in enumerate(state_list):
    meta_preds_verification[:,i] = state.meta_model.predict_proba(ensemble.features_test)[:,1]
np.all(meta_preds_verification==ensemble_meta_preds)

np.True_

In [32]:
first_model_best = (np.argmax(ensemble_meta_preds,axis=1)==0) # all indices where first model most confident
first_model_preds = state_list[0].model.predict(ensemble.features_test)
np.all(first_model_preds[first_model_best]==ensemble_preds[first_model_best])


np.True_

# Miscellaneous Testing

In [11]:
importlib.reload(policies)

pol = policies.SingleCoordinateExpert(3,2)
print(pol.target_coordinate)
preds = np.zeros((5,3))
pol.run_given_preds(preds)


2


array([[0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.]])